In [1]:
# Cell 1: environment check
import os
import random
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPU available: []


In [2]:
# Cell 2: point to the data and list the classes
# Assumes one folder per team, e.g. data/team_a/*.jpg, data/team_b/*.jpg
from pathlib import Path
from collections import Counter

DATA_DIR = Path("../data/Seria A teams/")  # change this to your folder

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
records = []
for team_dir in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    for img_path in team_dir.rglob("*"):
        if img_path.suffix.lower() in IMG_EXTS:
            records.append((str(img_path), team_dir.name))

print(f"Total images: {len(records)}")
print("Per team:")
for team, count in Counter(t for _, t in records).most_common():
    print(f"  {team}: {count}")

Total images: 2000
Per team:
  AC Milan: 100
  AC Monza: 100
  Atalanta: 100
  Bologna: 100
  Cagliari: 100
  Como 1907: 100
  Empoli: 100
  Fiorentina: 100
  Genoa: 100
  Inter Milan: 100
  Juventus: 100
  Lazio: 100
  Napoli: 100
  Parma: 100
  Roma: 100
  Torino: 100
  Udinese: 100
  US Lecce: 100
  Venezia FC: 100
  Verona: 100


In [3]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder

def get_group(img_path: str):
    # Assumes sessions are subfolders: data/team_a/session_01/*.jpg
    parts = Path(img_path).relative_to(DATA_DIR).parts
    return "/".join(parts[:2]) if len(parts) > 2 else None

paths = np.array([r[0] for r in records])
team_names = np.array([r[1] for r in records])

le = LabelEncoder()
y = le.fit_transform(team_names)
CLASS_NAMES = list(le.classes_)

raw_groups = [get_group(p) for p in paths]
n_ungrouped = sum(g is None for g in raw_groups)
if n_ungrouped:
    print(f"WARNING: {n_ungrouped} images have no session folder. "
          "Each is its own group, so near-duplicates may leak across splits.")
groups = np.array([g if g is not None else p for g, p in zip(raw_groups, paths)])

# Hold out ~20% as test, then ~20% of the rest as validation
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(outer.split(paths, y, groups))

inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
tr_rel, va_rel = next(inner.split(paths[trainval_idx], y[trainval_idx], groups[trainval_idx]))
train_idx = trainval_idx[tr_rel]
val_idx = trainval_idx[va_rel]

# No session may appear in two splits
assert set(groups[train_idx]).isdisjoint(groups[val_idx])
assert set(groups[trainval_idx]).isdisjoint(groups[test_idx])

for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    print(f"{name}: {len(idx)} images", dict(Counter(CLASS_NAMES[i] for i in y[idx])))

train: 1200 images {np.str_('AC Milan'): 60, np.str_('AC Monza'): 60, np.str_('Atalanta'): 60, np.str_('Bologna'): 60, np.str_('Cagliari'): 60, np.str_('Como 1907'): 60, np.str_('Empoli'): 60, np.str_('Fiorentina'): 60, np.str_('Genoa'): 60, np.str_('Inter Milan'): 60, np.str_('Juventus'): 60, np.str_('Lazio'): 60, np.str_('Napoli'): 60, np.str_('Parma'): 60, np.str_('Roma'): 60, np.str_('Torino'): 60, np.str_('Udinese'): 60, np.str_('US Lecce'): 60, np.str_('Venezia FC'): 60, np.str_('Verona'): 60}
val: 400 images {np.str_('AC Milan'): 20, np.str_('AC Monza'): 20, np.str_('Atalanta'): 20, np.str_('Bologna'): 20, np.str_('Cagliari'): 20, np.str_('Como 1907'): 20, np.str_('Empoli'): 20, np.str_('Fiorentina'): 20, np.str_('Genoa'): 20, np.str_('Inter Milan'): 20, np.str_('Juventus'): 20, np.str_('Lazio'): 20, np.str_('Napoli'): 20, np.str_('Parma'): 20, np.str_('Roma'): 20, np.str_('Torino'): 20, np.str_('Udinese'): 20, np.str_('US Lecce'): 20, np.str_('Venezia FC'): 20, np.str_('Verona'

In [4]:
# Cell 4: tf.data input pipeline with augmentation
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE = (160, 160)
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

# Horizontal flip mirrors jersey numbers and text. Remove it if numbers matter.
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="augment")

def make_ds(idx, training):
    ds = tf.data.Dataset.from_tensor_slices((paths[idx].tolist(), y[idx]))
    if training:
        ds = ds.shuffle(len(idx), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, l: (augment(x, training=True), l), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_idx, training=True)
val_ds = make_ds(val_idx, training=False)
test_ds = make_ds(test_idx, training=False)  # not shuffled, so order matches test_idx

# Weight each class inversely to its frequency in the training set
weights = compute_class_weight("balanced", classes=np.arange(len(CLASS_NAMES)), y=y[train_idx])
class_weight = dict(enumerate(weights))
print("Class weights:", {CLASS_NAMES[k]: round(v, 2) for k, v in class_weight.items()})

Class weights: {np.str_('AC Milan'): np.float64(1.0), np.str_('AC Monza'): np.float64(1.0), np.str_('Atalanta'): np.float64(1.0), np.str_('Bologna'): np.float64(1.0), np.str_('Cagliari'): np.float64(1.0), np.str_('Como 1907'): np.float64(1.0), np.str_('Empoli'): np.float64(1.0), np.str_('Fiorentina'): np.float64(1.0), np.str_('Genoa'): np.float64(1.0), np.str_('Inter Milan'): np.float64(1.0), np.str_('Juventus'): np.float64(1.0), np.str_('Lazio'): np.float64(1.0), np.str_('Napoli'): np.float64(1.0), np.str_('Parma'): np.float64(1.0), np.str_('Roma'): np.float64(1.0), np.str_('Torino'): np.float64(1.0), np.str_('US Lecce'): np.float64(1.0), np.str_('Udinese'): np.float64(1.0), np.str_('Venezia FC'): np.float64(1.0), np.str_('Verona'): np.float64(1.0)}


In [5]:
# Cell 5: CNN architecture
def conv_block(x, filters):
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    return tf.keras.layers.Dropout(0.25)(x)

def build_cnn(num_classes):
    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x = inputs
    for filters in [32, 64, 128, 256]:
        x = conv_block(x, filters)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="team_cnn")

model = build_cnn(len(CLASS_NAMES))
model.summary()

Model: "team_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 160, 160, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 160, 160, 32)        │             864 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 160, 160, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation (Activation)              │ (None, 160, 160, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 160, 160, 32)        │           9,216 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 160, 160, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_1 (Activation)            │ (None, 160, 160, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 80, 80, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 80, 80, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 80, 80, 64)          │          18,432 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 80, 80, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_2 (Activation)            │ (None, 80, 80, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 80, 80, 64)          │          36,864 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 80, 80, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_3 (Activation)            │ (None, 80, 80, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 40, 40, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 40, 40, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 40, 40, 128)         │          73,728 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 40, 40, 128)         │             512 │
│ (BatchNormalization)                 │                             │              

 Total params: 1,210,612 (4.62 MB)

 Trainable params: 1,208,692 (4.61 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [7]:
# Cell 6: compile and train
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
    tf.keras.callbacks.ModelCheckpoint("best_team_cnn.keras", monitor="val_loss", save_best_only=True),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight,
    callbacks=callbacks,
)

Epoch 1/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 557s 14s/step - accuracy: 0.8792 - loss: 0.4362 - val_accuracy: 0.1000 - val_loss: 12.0870 - learning_rate: 0.0010
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 994s 26s/step - accuracy: 0.9475 - loss: 0.2213 - val_accuracy: 0.1500 - val_loss: 9.0079 - learning_rate: 0.0010
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 5163s 139s/step - accuracy: 0.9475 - loss: 0.1747 - val_accuracy: 0.2500 - val_loss: 7.7393 - learning_rate: 0.0010
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 273s 7s/step - accuracy: 0.9817 - loss: 0.0942 - val_accuracy: 0.2000 - val_loss: 6.1520 - learning_rate: 0.0010
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 517s 12s/step - accuracy: 0.9925 - loss: 0.0498 - val_accuracy: 0.3500 - val_loss: 5.9746 - learning_rate: 0.0010
Epoch 6/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 504s 13s/step - accuracy: 0.9883 - loss: 0.0529 - val_accuracy: 0.3500 - val_loss: 5.4315 - learning_rate: 0.0010
Epoch 7/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 439s 11s/step - accuracy: 0.9717 - loss: 0.1085 

In [8]:
model.save("../model/seriateams.keras")